# ISI Distribution Deep-Dive: Why Sequence Composition Affects the Non-Monotonicity

Investigation C showed that d'(ISI=1) vs sigma1 is **non-monotonic** in ISI=[1]-only
sequences but **monotonic** in ISI=[1,2,4] mixed sequences. This notebook investigates
**why** sequence composition has this effect.

| Section | Question |
|---|---|
| 2. Sequence Statistics | What structural differences exist between single-ISI and multi-ISI sequences? |
| 3. Bank Size | How big is the memory bank when ISI=1 repeats arrive? |
| 4. Controlled Experiment | Does increasing bank size (longer sequences) suppress the bump? |
| 5. Per-Trial Scores | How do hit/FA scores vary with trial position (= bank size)? |
| 6. FA Pool | How does the FA distribution differ between sequence types? |

In [ ]:
import sys, os, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict
from scipy.spatial.distance import pdist

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.plotting import ensure_dir
from utls.loading import (
    load_results_with_exclusion_2,
    move_sequences_to_used,
    load_results_with_exclusion_no_dropping,
)
from utls.runners_v2 import run_experiment_scores, make_noise_schedule
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.io_utils import (
    make_model_save_dir,
    save_all_figures,
    save_single_figure,
    save_runs_summary,
)
from encoders import *

from utls.toy_experiments import (
    make_toy_experiment_list,
    make_compact_multi_isi_sequences,
    infer_trial_isis,
)
from utls.sigma_fitting import log_mid

from sklearn.metrics import roc_auc_score, roc_curve

try:
    get_ipython()
    from tqdm.notebook import tqdm, trange
except NameError:
    from tqdm import tqdm, trange

In [ ]:
# --- Load config, experiment data, and encode stimuli ---

def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path

CONFIG_PATH = (
    "/om2/user/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)
model_cfg, _ = load_config(CONFIG_PATH)

exp_cfg = model_cfg["experiment"]
which_task, is_multi = exp_cfg["which_task"], exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)
isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]

metric = model_cfg["metric"]
noise_cfg = model_cfg["noise_model"]
noise_mode, t_step = noise_cfg["name"], noise_cfg["t_step"]

repr_cfg = model_cfg["representation"]
time_avg = repr_cfg["time_avg"]
encoder_type = repr_cfg["type"]
layer = repr_cfg.get("layer", None)
pc_dims = repr_cfg.get("pc_dims", None)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)
human_curve = compute_human_curve(human_runs, is_multi, which_isi)

# Encode stimuli
NN_ENCODERS = {"kell2018", "resnet50"}
encoder_task = "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
encoder_cfg = dict(
    encoder_type=encoder_type, model_name=encoder_type, task=encoder_task,
    statistics_dict=statistics_set.statistics, model_params=model_params,
    pc_dims=pc_dims, sr=20000, duration=2.0, rms_level=0.05,
    time_avg=time_avg, device="cuda",
)
if encoder_type in NN_ENCODERS:
    encoder_cfg["layer"] = layer
encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)

# Param bounds and human targets
param_bounds = {"sigma0": (3.0, 22), "sigma1": (0.01, 10.0), "sigma2": (0.0005, 0.5)}
sigma1_human = {isi: float(human_curve[i]) for i, isi in enumerate(isis) if isi in [1, 2, 4]}
sigma2_init = log_mid(*param_bounds["sigma2"])

stimulus_pool = sorted({s for seq in exp_list for s in seq})

print(f"ISIs: {isis}")
print(f"Human d' curve: {human_curve}")
print(f"Stimulus pool: {len(stimulus_pool)} items")
print(f"Encoded shape: {X.shape}")
print(f"sigma1 human targets: {sigma1_human}")

---
## 2. Sequence Statistics & Structure

Characterize structural differences between single-ISI and multi-ISI sequences.
What changes when you go from 1 ISI condition to several?

In [ ]:
# Generate four sequence types to compare
seq_configs = {}

# ISI=[1]-only, short (original from Investigation C)
e, k = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1],
    n_sequences=8, length=15, min_pairs_per_isi=5, seed=1000,
)
seq_configs["ISI=[1] short (L=15)"] = {"exps": e, "keys": k, "isi_values": [1]}

# ISI=[1]-only, long (same length as mixed)
e, k = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1],
    n_sequences=10, length=60, min_pairs_per_isi=5, seed=1001,
)
seq_configs["ISI=[1] long (L=60)"] = {"exps": e, "keys": k, "isi_values": [1]}

# ISI=[1,2,4] mixed
e, k = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1, 2, 4],
    n_sequences=10, length=60, min_pairs_per_isi=5, seed=3000,
)
seq_configs["ISI=[1,2,4] (L=60)"] = {"exps": e, "keys": k, "isi_values": [1, 2, 4]}

# ISI=[8,16,32,64] long-delay
e, k = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[8, 16, 32, 64],
    n_sequences=10, length=78, min_pairs_per_isi=5, seed=4000,
)
seq_configs["ISI=[8,16,32,64] (L=78)"] = {"exps": e, "keys": k, "isi_values": [8, 16, 32, 64]}

for name, cfg in seq_configs.items():
    print(f"{name}: {len(cfg['exps'])} seqs x {len(cfg['exps'][0])} trials")

In [ ]:
# Raster plot: visualize where pairs and foils land in each sequence type

def classify_positions(seq):
    """For each trial position, return (role, isi_value).
    role: 'foil', 'first_pres', or 'repeat'
    isi_value: ISI for pairs, -1 for foils
    """
    first_seen = {}
    repeat_of = {}  # position -> first_pos
    for i, stim in enumerate(seq):
        if stim in first_seen:
            repeat_of[i] = first_seen[stim]
        else:
            first_seen[stim] = i

    # Which first-presentations have a repeat later?
    first_with_repeat = set(repeat_of.values())

    roles = []
    for i, stim in enumerate(seq):
        if i in repeat_of:
            isi = i - repeat_of[i] - 1
            roles.append(('repeat', isi))
        elif i in first_with_repeat:
            # Find the repeat position to get ISI
            for rep_pos, fp in repeat_of.items():
                if fp == i:
                    isi = rep_pos - i - 1
                    roles.append(('first_pres', isi))
                    break
        else:
            roles.append(('foil', -1))
    return roles


isi_colors = {-1: 'lightgray', 1: 'tab:blue', 2: 'tab:orange', 4: 'tab:green',
              8: 'tab:red', 16: 'tab:purple', 32: 'tab:brown', 64: 'tab:pink'}

fig, axes = plt.subplots(len(seq_configs), 1, figsize=(16, 3 * len(seq_configs)))

for ax, (name, cfg) in zip(axes, seq_configs.items()):
    exps = cfg["exps"]
    for row, seq in enumerate(exps):
        roles = classify_positions(seq)
        for col, (role, isi) in enumerate(roles):
            color = isi_colors.get(isi, 'gray')
            alpha = 1.0 if role == 'repeat' else (0.4 if role == 'first_pres' else 0.15)
            marker = 'v' if role == 'repeat' else ('o' if role == 'first_pres' else 's')
            ax.scatter(col, row, c=color, alpha=alpha, s=20, marker=marker, edgecolors='none')

    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Trial position")
    ax.set_ylabel("Sequence #")
    ax.set_yticks(range(len(exps)))
    ax.set_xlim(-0.5, max(len(s) for s in exps) - 0.5)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='lightgray', alpha=0.3, ms=8, label='Foil'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:blue', alpha=0.5, ms=8, label='First pres (pair)'),
    Line2D([0], [0], marker='v', color='w', markerfacecolor='tab:blue', ms=8, label='Repeat'),
]
axes[0].legend(handles=legend_elements, loc='upper right', fontsize=8, frameon=False)

fig.suptitle("Sequence structure raster plots", y=1.01, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Comprehensive sequence statistics comparison

print(f"{'Config':<30s} {'Trials':>6s} {'Unique':>6s} {'Pairs':>6s} {'RepRate':>8s} "
      f"{'Foil/Rep':>8s} {'InterRepGap':>12s} {'1st ISI=1 pos':>14s}")
print("-" * 100)

for name, cfg in seq_configs.items():
    exps = cfg["exps"]
    n_trials = len(exps[0])

    unique_counts, pair_counts, rep_rates = [], [], []
    foil_counts, inter_rep_gaps, first_isi1_positions = [], [], []

    for seq in exps:
        roles = classify_positions(seq)
        n_unique = len(set(seq))
        n_repeats = sum(1 for r, _ in roles if r == 'repeat')
        n_foils = sum(1 for r, _ in roles if r == 'foil')
        n_pairs = n_repeats  # each pair has exactly 1 repeat

        unique_counts.append(n_unique)
        pair_counts.append(n_pairs)
        rep_rates.append(n_repeats / n_trials)
        foil_counts.append(n_foils)

        # Inter-repeat gap: distance between consecutive repeat events
        repeat_positions = [i for i, (r, _) in enumerate(roles) if r == 'repeat']
        if len(repeat_positions) > 1:
            gaps = np.diff(repeat_positions)
            inter_rep_gaps.extend(gaps.tolist())

        # First ISI=1 repeat position
        isi1_positions = [i for i, (r, isi) in enumerate(roles) if r == 'repeat' and isi == 1]
        if isi1_positions:
            first_isi1_positions.append(isi1_positions[0])

    irg = f"{np.mean(inter_rep_gaps):.1f}+/-{np.std(inter_rep_gaps):.1f}" if inter_rep_gaps else "N/A"
    fp = f"{np.mean(first_isi1_positions):.1f}+/-{np.std(first_isi1_positions):.1f}" if first_isi1_positions else "N/A"

    print(f"{name:<30s} {n_trials:>6d} {np.mean(unique_counts):>6.1f} {np.mean(pair_counts):>6.1f} "
          f"{np.mean(rep_rates):>8.3f} {np.mean(foil_counts)/max(np.mean(pair_counts),1):>8.2f} "
          f"{irg:>12s} {fp:>14s}")

    # Per-ISI pair counts
    isi_pair_counts = defaultdict(list)
    for seq in exps:
        trial_isis = infer_trial_isis(seq)
        counts = Counter(trial_isis)
        for isi_val in cfg["isi_values"]:
            isi_pair_counts[isi_val].append(counts.get(isi_val, 0))

    for isi_val in cfg["isi_values"]:
        vals = isi_pair_counts[isi_val]
        print(f"  ISI={isi_val}: {np.mean(vals):.1f} +/- {np.std(vals):.1f} pairs/seq")
    print()

In [ ]:
# Position distribution: where do ISI=1 repeats, first-presentations, and foils land?

configs_with_isi1 = {k: v for k, v in seq_configs.items() if 1 in v["isi_values"]}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ["ISI=1 repeat positions", "ISI=1 first-pres positions", "Foil positions"]

for name, cfg in configs_with_isi1.items():
    isi1_rep_pos, isi1_fp_pos, foil_pos = [], [], []
    for seq in cfg["exps"]:
        roles = classify_positions(seq)
        for i, (role, isi) in enumerate(roles):
            if role == 'repeat' and isi == 1:
                isi1_rep_pos.append(i)
            elif role == 'first_pres' and isi == 1:
                isi1_fp_pos.append(i)
            elif role == 'foil':
                foil_pos.append(i)

    max_len = len(cfg["exps"][0])
    bins = np.arange(-0.5, max_len + 0.5, 1)
    axes[0].hist(isi1_rep_pos, bins=bins, alpha=0.5, density=True, label=name)
    axes[1].hist(isi1_fp_pos, bins=bins, alpha=0.5, density=True, label=name)
    axes[2].hist(foil_pos, bins=bins, alpha=0.5, density=True, label=name)

for ax, title in zip(axes, titles):
    ax.set_xlabel("Trial position")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(fontsize=7, frameon=False)
    ax.grid(alpha=0.25)

fig.suptitle("Position distributions for ISI=1 events", y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Memory bank composition at ISI=1 repeat time
# For each ISI=1 repeat, classify what's in the bank:
# (a) target match, (b) first-pres waiting for repeat, (c) already-repeated, (d) true foil

def bank_composition_at_repeats(seq, target_isi=1):
    """For each ISI=target_isi repeat, return the bank composition."""
    first_seen = {}
    repeat_of = {}
    for i, stim in enumerate(seq):
        if stim in first_seen:
            repeat_of[i] = first_seen[stim]
        else:
            first_seen[stim] = i

    # Build lookup: which positions are first-pres of pairs?
    first_pres_of_pair = set(repeat_of.values())
    # Map first_pres -> repeat_pos
    fp_to_rep = {fp: rp for rp, fp in repeat_of.items()}

    compositions = []
    for rep_pos, fp_pos in sorted(repeat_of.items()):
        isi = rep_pos - fp_pos - 1
        if isi != target_isi:
            continue

        # At trial rep_pos, the bank contains items from positions 0..rep_pos-1
        # (each item is stored when first seen)
        n_match = 0
        n_waiting = 0      # first-pres whose repeat hasn't happened yet
        n_already_repeated = 0  # items where both pres have occurred
        n_foil = 0          # items that never repeat

        for prev_pos in range(rep_pos):
            stim = seq[prev_pos]
            if prev_pos != first_seen.get(stim):
                continue  # this is a repeat presentation, already counted via first_seen

            if prev_pos == fp_pos:
                n_match += 1
            elif prev_pos in first_pres_of_pair:
                their_rep = fp_to_rep[prev_pos]
                if their_rep < rep_pos:
                    n_already_repeated += 1
                else:
                    n_waiting += 1
            else:
                n_foil += 1

        compositions.append({
            'bank_size': rep_pos,  # bank has rep_pos items (0-indexed trial positions)
            'match': n_match,
            'waiting': n_waiting,
            'already_repeated': n_already_repeated,
            'foil': n_foil,
        })
    return compositions


fig, axes = plt.subplots(1, len(configs_with_isi1), figsize=(5 * len(configs_with_isi1), 5))
if len(configs_with_isi1) == 1:
    axes = [axes]

categories = ['match', 'waiting', 'already_repeated', 'foil']
cat_colors = ['tab:green', 'tab:orange', 'tab:blue', 'lightgray']
cat_labels = ['Target match', 'Waiting for repeat', 'Already repeated', 'True foil']

for ax, (name, cfg) in zip(axes, configs_with_isi1.items()):
    all_comps = []
    for seq in cfg["exps"]:
        all_comps.extend(bank_composition_at_repeats(seq, target_isi=1))

    if not all_comps:
        ax.set_title(f"{name}\n(no ISI=1 repeats)")
        continue

    # Average composition
    means = {cat: np.mean([c[cat] for c in all_comps]) for cat in categories}
    total = sum(means.values())

    bottom = 0
    for cat, color, label in zip(categories, cat_colors, cat_labels):
        ax.bar(0, means[cat], bottom=bottom, color=color, label=label, width=0.6)
        if means[cat] > 0.3:
            ax.text(0, bottom + means[cat] / 2, f"{means[cat]:.1f}", ha='center', va='center', fontsize=10)
        bottom += means[cat]

    ax.set_title(f"{name}\nbank size = {total:.1f}", fontsize=10)
    ax.set_ylabel("# items in bank")
    ax.set_xticks([])
    ax.legend(fontsize=8, frameon=False, loc='upper right')

fig.suptitle("Memory bank composition when ISI=1 repeat arrives", y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

---
## 3. Memory Bank Size at Repeat Time

The memory bank grows by 1 item per trial (bank size at trial t = t).
How big is the bank when ISI=1 repeats arrive?

In [ ]:
# Bank size histograms and box plots at repeat time

def get_bank_sizes_at_repeats(seqs):
    """For each repeat in each sequence, return (isi, bank_size, trial_pos)."""
    records = []
    for seq in seqs:
        first_seen = {}
        for i, stim in enumerate(seq):
            if stim in first_seen:
                isi = i - first_seen[stim] - 1
                records.append((isi, i, i))  # bank_size = trial_pos (0-indexed)
            else:
                first_seen[stim] = i
    return records

# Collect bank sizes for ISI=1 repeats across all configs with ISI=1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of bank sizes at ISI=1 repeat time
for name, cfg in configs_with_isi1.items():
    records = get_bank_sizes_at_repeats(cfg["exps"])
    isi1_bank_sizes = [bs for isi, bs, _ in records if isi == 1]
    axes[0].hist(isi1_bank_sizes, bins=np.arange(-0.5, 60.5, 1), alpha=0.5, density=True, label=name)
axes[0].set_xlabel("Bank size at ISI=1 repeat")
axes[0].set_ylabel("Density")
axes[0].set_title("Bank size when ISI=1 repeat arrives")
axes[0].legend(fontsize=8, frameon=False)
axes[0].grid(alpha=0.25)

# Right: box plot comparing bank sizes
box_data, box_labels = [], []
for name, cfg in configs_with_isi1.items():
    records = get_bank_sizes_at_repeats(cfg["exps"])
    isi1_bank_sizes = [bs for isi, bs, _ in records if isi == 1]
    box_data.append(isi1_bank_sizes)
    box_labels.append(name.replace(" ", "\n"))

bp = axes[1].boxplot(box_data, labels=box_labels, patch_artist=True)
colors = ['tab:blue', 'tab:orange', 'tab:green']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[1].set_ylabel("Bank size")
axes[1].set_title("Bank size at ISI=1 repeat time")
axes[1].grid(alpha=0.25, axis='y')

# Print summary
for name, data in zip(configs_with_isi1.keys(), box_data):
    print(f"{name}: bank size = {np.mean(data):.1f} +/- {np.std(data):.1f} "
          f"(range {min(data)}-{max(data)})")

fig.tight_layout()
plt.show()

---
## 4. Controlled Bank-Size Experiment

**Key test**: Generate ISI=[1]-only sequences at increasing lengths (15, 30, 45, 60).
Longer sequences = more foils = larger memory bank at repeat time.
If the non-monotonic bump disappears as length increases, **bank size is the mechanism**.

In [ ]:
# sweep_sigma1_detailed: evaluate one sigma1 value, return d', AUROC, MSE, raw scores

def sweep_sigma1_detailed(
    sigma1_value, sigma0, sigma2, t_step,
    noise_mode, metric, X0, name_to_idx,
    experiment_list, target_isis,
    human_dprimes_by_isi=None,
    n_mc=64, seed=0,
):
    seq_trial_isis = [infer_trial_isis(seq) for seq in experiment_list]
    all_hits_by_isi = {isi: [] for isi in target_isis}
    all_fas = []

    for rep in range(n_mc):
        for si, seq in enumerate(experiment_list):
            t_isis = seq_trial_isis[si]
            run_out = run_experiment_scores(
                sigma0=sigma0, sigma1=sigma1_value, sigma2=sigma2,
                t_step=t_step, rate=0, noise_mode=noise_mode,
                metric=metric, X0=X0, name_to_idx=name_to_idx,
                experiment_list=[seq], debug=False,
                seed=seed + rep * 1000 + si,
            )
            h = np.asarray(run_out["hits"])
            f = np.asarray(run_out["fas"])
            if len(h) != len(t_isis):
                continue
            for isi_val in target_isis:
                mask = np.array(t_isis) == isi_val
                all_hits_by_isi[isi_val].extend(h[mask].tolist())
            all_fas.extend(f.tolist())

    fas_arr = np.array(all_fas)
    result = {
        "sigma1": sigma1_value,
        "dprime_by_isi": {}, "auroc_by_isi": {}, "mse_by_isi": {},
        "hit_scores_by_isi": {}, "fa_scores": fas_arr,
    }
    for isi_val in target_isis:
        hits_isi = np.array(all_hits_by_isi[isi_val])
        result["hit_scores_by_isi"][isi_val] = hits_isi
        if len(hits_isi) == 0 or len(fas_arr) == 0:
            result["dprime_by_isi"][isi_val] = np.nan
            result["auroc_by_isi"][isi_val] = np.nan
            result["mse_by_isi"][isi_val] = np.nan
            continue
        y_true = np.concatenate([np.ones(len(hits_isi)), np.zeros(len(fas_arr))])
        scores = np.concatenate([hits_isi, fas_arr])
        auroc = roc_auc_score(y_true, -scores)
        dprime = auroc_to_dprime(auroc)
        result["auroc_by_isi"][isi_val] = auroc
        result["dprime_by_isi"][isi_val] = dprime
        if human_dprimes_by_isi and isi_val in human_dprimes_by_isi:
            result["mse_by_isi"][isi_val] = (dprime - human_dprimes_by_isi[isi_val]) ** 2
        else:
            result["mse_by_isi"][isi_val] = np.nan
    return result


SIGMA1_GRID = np.exp(np.linspace(np.log(0.01), np.log(10.0), 30))
N_MC = 64

def run_sigma1_sweep(experiment_list, sigma0=13.5, desc="sweep", seed=0):
    results = []
    for i in trange(len(SIGMA1_GRID), desc=desc):
        r = sweep_sigma1_detailed(
            sigma1_value=SIGMA1_GRID[i], sigma0=sigma0, sigma2=sigma2_init,
            t_step=t_step, noise_mode=noise_mode, metric=metric,
            X0=X, name_to_idx=name_to_idx, experiment_list=experiment_list,
            target_isis=[1], human_dprimes_by_isi=sigma1_human,
            n_mc=N_MC, seed=seed + i * 100_000,
        )
        results.append(r)
    return results

print(f"Sigma1 grid: {len(SIGMA1_GRID)} points, [{SIGMA1_GRID[0]:.4f}, {SIGMA1_GRID[-1]:.4f}]")

In [ ]:
# Generate ISI=[1]-only sequences at intermediate lengths (30, 45)
length_configs = {}

# Reuse length=15 and length=60 from seq_configs
length_configs[15] = seq_configs["ISI=[1] short (L=15)"]["exps"]
length_configs[60] = seq_configs["ISI=[1] long (L=60)"]["exps"]

for L in [30, 45]:
    e, _ = make_compact_multi_isi_sequences(
        stimulus_pool=stimulus_pool, isi_values=[1],
        n_sequences=10, length=L, min_pairs_per_isi=5, seed=5000 + L,
    )
    length_configs[L] = e
    print(f"ISI=[1] L={L}: {len(e)} seqs x {len(e[0])} trials")

# Also include mixed sequences for comparison
length_configs["mixed"] = seq_configs["ISI=[1,2,4] (L=60)"]["exps"]

# Verify bank sizes at ISI=1 repeat for each length
print("\nBank size at ISI=1 repeat time:")
for label, exps in sorted(length_configs.items(), key=lambda x: str(x[0])):
    records = get_bank_sizes_at_repeats(exps)
    isi1_bs = [bs for isi, bs, _ in records if isi == 1]
    if isi1_bs:
        print(f"  L={label}: {np.mean(isi1_bs):.1f} +/- {np.std(isi1_bs):.1f} (n={len(isi1_bs)})")

In [ ]:
# Run sigma1 sweeps for each sequence length
sweep_by_length = {}

for label in [15, 30, 45, 60, "mixed"]:
    desc = f"L={label}" if isinstance(label, int) else "ISI=[1,2,4]"
    print(f"\n--- {desc} ---")
    sweep_by_length[label] = run_sigma1_sweep(
        length_configs[label], sigma0=13.5,
        desc=desc, seed=500_000 + (label if isinstance(label, int) else 99),
    )

In [ ]:
# KEY PLOT: d'(ISI=1) vs sigma1, one curve per sequence length

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.viridis

labels_ordered = [15, 30, 45, 60, "mixed"]
colors = cmap(np.linspace(0.05, 0.95, len(labels_ordered)))

for color, label in zip(colors, labels_ordered):
    results = sweep_by_length[label]
    s1 = np.array([r["sigma1"] for r in results])
    dp = np.array([r["dprime_by_isi"].get(1, np.nan) for r in results])
    auc = np.array([r["auroc_by_isi"].get(1, np.nan) for r in results])

    lbl = f"ISI=[1] L={label}" if isinstance(label, int) else "ISI=[1,2,4] L=60"
    axes[0].plot(s1, dp, "o-", ms=3, color=color, label=lbl)
    axes[1].plot(s1, auc, "o-", ms=3, color=color, label=lbl)

axes[0].axhline(sigma1_human[1], ls=":", color="red", alpha=0.7, label="human d'")

for ax, ylabel, title in zip(axes, ["d'", "AUROC"],
                              ["d'(ISI=1) vs sigma1", "AUROC(ISI=1) vs sigma1"]):
    ax.set_xscale("log")
    ax.set_xlabel("sigma1")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.25)

fig.suptitle("Section 4: Effect of sequence length (bank size) on non-monotonicity\n"
             "If bump flattens as L increases, bank size is the mechanism",
             y=1.05, fontsize=12)
fig.tight_layout()
plt.show()

---
## 5. Per-Trial-Position Score Analysis

Use `isi_hit_dists` and `fa_by_t` from the model runner to see how hit and FA
scores vary with trial position (= bank size).

NOTE: `isi_hit_dists` uses ISI = t - last_seen (gap inclusive). ISI=1 (1
intervening item) maps to **key 2** in `isi_hit_dists`.

In [ ]:
# Collect per-trial hit and FA scores at a few sigma1 values

ISI_KEY_FOR_ISI1 = 2  # isi_hit_dists convention: ISI=1 -> key 2

def collect_per_trial_scores(experiment_list, sigma1, sigma0=13.5, n_mc=64, seed=0):
    """Run model and collect (score, trial_position) for hits and FAs."""
    all_hit_score_pos = []  # (score, trial_pos) for ISI=1 hits
    all_fa_score_pos = []   # (score, trial_pos) for FAs

    for rep in range(n_mc):
        run_out = run_experiment_scores(
            sigma0=sigma0, sigma1=sigma1, sigma2=sigma2_init,
            t_step=t_step, rate=0, noise_mode=noise_mode,
            metric=metric, X0=X, name_to_idx=name_to_idx,
            experiment_list=experiment_list, debug=False,
            seed=seed + rep,
        )
        # ISI=1 hits: key=2 in isi_hit_dists
        for score, t_pos in run_out["isi_hit_dists"].get(ISI_KEY_FOR_ISI1, []):
            all_hit_score_pos.append((score, t_pos))
        # FAs by trial position
        for t_pos, fa_scores in run_out["fa_by_t"].items():
            for score in fa_scores:
                all_fa_score_pos.append((score, t_pos))

    return all_hit_score_pos, all_fa_score_pos


# Three sigma1 values: low, bump-peak, high
SIGMA1_DIAG = [0.05, 0.3, 3.0]

per_trial = {}
for label in ["ISI=[1] short (L=15)", "ISI=[1] long (L=60)"]:
    per_trial[label] = {}
    for s1 in SIGMA1_DIAG:
        print(f"  {label}, sigma1={s1}...")
        hits, fas = collect_per_trial_scores(
            seq_configs[label]["exps"], sigma1=s1, seed=600_000 + int(s1 * 100),
        )
        per_trial[label][s1] = {"hits": hits, "fas": fas}

print("Done collecting per-trial scores.")

In [ ]:
# Scatter plots: score vs trial position for hits and FAs

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
seq_labels = ["ISI=[1] short (L=15)", "ISI=[1] long (L=60)"]

for row, label in enumerate(seq_labels):
    for col, s1 in enumerate(SIGMA1_DIAG):
        ax = axes[row, col]
        data = per_trial[label][s1]
        hits = data["hits"]
        fas = data["fas"]

        if fas:
            fa_scores, fa_pos = zip(*fas)
            ax.scatter(fa_pos, fa_scores, alpha=0.05, s=5, c='tab:blue', label='FA')
        if hits:
            h_scores, h_pos = zip(*hits)
            ax.scatter(h_pos, h_scores, alpha=0.15, s=10, c='tab:orange', label='Hit (ISI=1)')

        ax.set_xlabel("Trial position (= bank size)")
        ax.set_ylabel("Distance score (lower = more similar)")
        ax.set_title(f"{label}\nsigma1={s1}")
        ax.legend(fontsize=7, frameon=False, loc='upper right')
        ax.grid(alpha=0.25)

fig.suptitle("Section 5: Hit and FA scores vs trial position", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Mean score vs trial position (binned) with std bands

def binned_scores(score_pos_list, n_bins=10):
    """Bin scores by trial position and return bin centers, means, stds."""
    if not score_pos_list:
        return [], [], []
    scores, positions = zip(*score_pos_list)
    scores, positions = np.array(scores), np.array(positions)
    bin_edges = np.linspace(positions.min(), positions.max() + 1, n_bins + 1)
    centers, means, stds = [], [], []
    for i in range(n_bins):
        mask = (positions >= bin_edges[i]) & (positions < bin_edges[i + 1])
        if mask.sum() > 0:
            centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            means.append(np.mean(scores[mask]))
            stds.append(np.std(scores[mask]))
    return np.array(centers), np.array(means), np.array(stds)


fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, label in enumerate(seq_labels):
    for col, s1 in enumerate(SIGMA1_DIAG):
        ax = axes[row, col]
        data = per_trial[label][s1]

        # Hits
        hc, hm, hs = binned_scores(data["hits"], n_bins=8)
        if len(hc) > 0:
            ax.plot(hc, hm, 'o-', color='tab:orange', label='Hit mean (ISI=1)')
            ax.fill_between(hc, hm - hs, hm + hs, alpha=0.2, color='tab:orange')

        # FAs
        fc, fm, fs = binned_scores(data["fas"], n_bins=8)
        if len(fc) > 0:
            ax.plot(fc, fm, 's-', color='tab:blue', label='FA mean')
            ax.fill_between(fc, fm - fs, fm + fs, alpha=0.2, color='tab:blue')

        ax.set_xlabel("Trial position (= bank size)")
        ax.set_ylabel("Mean distance score")
        ax.set_title(f"{label}, sigma1={s1}")
        ax.legend(fontsize=7, frameon=False)
        ax.grid(alpha=0.25)

fig.suptitle("Section 5: Mean hit/FA score vs trial position (binned)\n"
             "If FA scores decrease with position (larger bank), that explains d' drop",
             y=1.05, fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# Local AUROC: split by bank size (early vs late in sequence)

print("Local AUROC by bank size group:\n")
print(f"{'Sequence':<25s} {'sigma1':>7s} {'Group':>12s} {'AUROC':>7s} {'d\\'':>7s} {'n_hits':>7s} {'n_fas':>7s}")
print("-" * 80)

for label in seq_labels:
    for s1 in SIGMA1_DIAG:
        data = per_trial[label][s1]
        hits = data["hits"]
        fas = data["fas"]

        if not hits or not fas:
            continue

        h_scores, h_pos = zip(*hits)
        f_scores, f_pos = zip(*fas)
        h_scores, h_pos = np.array(h_scores), np.array(h_pos)
        f_scores, f_pos = np.array(f_scores), np.array(f_pos)

        # Define bank-size groups based on trial position
        thresholds = [(0, 5, "bank<5"), (5, 15, "bank 5-15"), (15, 999, "bank>15")]

        for lo, hi, group_name in thresholds:
            h_mask = (h_pos >= lo) & (h_pos < hi)
            f_mask = (f_pos >= lo) & (f_pos < hi)
            n_h = h_mask.sum()
            n_f = f_mask.sum()
            if n_h < 5 or n_f < 5:
                continue

            y_true = np.concatenate([np.ones(n_h), np.zeros(n_f)])
            scores = np.concatenate([h_scores[h_mask], f_scores[f_mask]])
            auroc = roc_auc_score(y_true, -scores)
            dp = auroc_to_dprime(auroc)

            print(f"{label:<25s} {s1:>7.2f} {group_name:>12s} {auroc:>7.4f} {dp:>7.3f} {n_h:>7d} {n_f:>7d}")

---
## 6. FA Pool Composition Analysis

The FA score = min distance to ANY memory in the bank. With a larger bank,
there are more candidates, so the minimum is likely smaller (order statistics
effect). This compresses the FA distribution toward zero, making it harder to
distinguish from hits.

In [ ]:
# FA distribution comparison: short vs long vs mixed sequences at bump-peak sigma1

s1_bump = 0.3  # approximate bump peak

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Overall FA distributions
for label in ["ISI=[1] short (L=15)", "ISI=[1] long (L=60)"]:
    fas = per_trial[label][s1_bump]["fas"]
    if fas:
        fa_scores = [s for s, _ in fas]
        axes[0].hist(fa_scores, bins=50, alpha=0.5, density=True, label=label)

axes[0].set_xlabel("FA distance score")
axes[0].set_ylabel("Density")
axes[0].set_title(f"FA distributions at sigma1={s1_bump}")
axes[0].legend(fontsize=8, frameon=False)
axes[0].grid(alpha=0.25)

# Panel 2: FA scores stratified by trial position (early vs late)
for label, color_base in zip(seq_labels, ['tab:blue', 'tab:orange']):
    fas = per_trial[label][s1_bump]["fas"]
    if not fas:
        continue
    f_scores, f_pos = zip(*fas)
    f_scores, f_pos = np.array(f_scores), np.array(f_pos)

    early = f_scores[f_pos < 5]
    late = f_scores[f_pos >= 10]

    short_name = "L=15" if "short" in label else "L=60"
    if len(early) > 0:
        axes[1].hist(early, bins=40, alpha=0.4, density=True,
                     label=f"{short_name} early (pos<5)", linestyle='--')
    if len(late) > 0:
        axes[1].hist(late, bins=40, alpha=0.4, density=True,
                     label=f"{short_name} late (pos>=10)")

axes[1].set_xlabel("FA distance score")
axes[1].set_ylabel("Density")
axes[1].set_title(f"FA by trial position at sigma1={s1_bump}")
axes[1].legend(fontsize=7, frameon=False)
axes[1].grid(alpha=0.25)

fig.suptitle("Section 6: FA pool composition\n"
             "Longer sequences should have FA distribution shifted left (lower distances)",
             y=1.05, fontsize=12)
fig.tight_layout()
plt.show()

# Print FA statistics
print("\nFA distribution statistics:")
for label in seq_labels:
    fas = per_trial[label][s1_bump]["fas"]
    if fas:
        fa_scores = np.array([s for s, _ in fas])
        print(f"  {label}: mean={np.mean(fa_scores):.4f}, "
              f"std={np.std(fa_scores):.4f}, "
              f"median={np.median(fa_scores):.4f}, "
              f"min={np.min(fa_scores):.4f}")

---
## 7. Summary

**Fill in after running:**

1. **Sequence statistics**: ISI=[1] short has ___ trials, ___% repeats. ISI=[1] long has ___ trials, ___% repeats. ISI=[1,2,4] mixed has ___ trials, ___% repeats.

2. **Bank size at ISI=1 repeat**: ISI=[1] short: bank ~ ___. ISI=[1] long: bank ~ ___. ISI=[1,2,4] mixed: bank ~ ___.

3. **Bank composition**: In short sequences, the bank at ISI=1 repeat contains mostly ___. In long sequences, it contains mostly ___.

4. **Controlled experiment** (Section 4): As sequence length increases from 15 to 60, the d' bump [does/does not] diminish. The mixed ISI=[1,2,4] curve [does/does not] match the ISI=[1] L=60 curve.

5. **Per-trial analysis** (Section 5): FA scores [do/do not] decrease with trial position. Hit scores [do/do not] change with trial position.

6. **FA pool** (Section 6): In longer sequences, the FA distribution is [shifted left / unchanged]. This [confirms / contradicts] the order-statistics mechanism.

**Conclusion**: The non-monotonicity in d'(ISI=1) vs sigma1 is [primarily caused by / unrelated to] the memory bank size. The min() decision rule over a small bank (2 items) behaves qualitatively differently from min() over a large bank (10+ items) because [mechanism].

---
## 3. Memory Bank Size at Repeat Time

Bank size = trial_position (0-indexed). The model stores one item per trial, so by trial t the bank has t items.